In [ ]:
# ======================================
# 2_similarity_labeling.ipynb
# ======================================

import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ------------------------------------
# Dataset for unlabeled reconstruction
# ------------------------------------
class ReconstructedDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = self._get_image_paths()

    def _get_image_paths(self):
        paths = []
        for root, _, files in os.walk(self.root_dir):
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    paths.append(os.path.join(root, f))
        return paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        p = self.image_paths[idx]
        im = Image.open(p).convert('L')
        if self.transform:
            im = self.transform(im)
        return im, p

# --------------------------
# Similarity functions
# --------------------------
def cosine_similarity(a, b):
    a = F.normalize(a, dim=-1)
    b = F.normalize(b, dim=-1)
    return torch.matmul(a, b.t())

def euclidean_similarity(a, b):
    a2 = (a**2).sum(dim=1, keepdim=True)
    b2 = (b**2).sum(dim=1, keepdim=True).t()
    d2 = a2 + b2 - 2*torch.matmul(a, b.t())
    d2 = torch.clamp(d2, min=0.0)
    d = torch.sqrt(d2 + 1e-12)
    return 1.0 / (1.0 + d)  # smaller distances → higher similarity

def combined_similarity(f, Tm, alpha=0.5):
    cs = cosine_similarity(f, Tm)
    es = euclidean_similarity(f, Tm)
    return alpha * cs + (1.0 - alpha) * es

# --------------------------
# Load template library
# --------------------------
template_lib_path = 'stage2_outputs/template_library.pt'  # saved from Notebook 1
ckpt = torch.load(template_lib_path, map_location=device)
Tm = ckpt['templates'].to(device)  # tensor of template features
class_names = ckpt['classes']
print("Loaded template library:", class_names)

# --------------------------
# Load feature extractor 
# --------------------------
transform_for_extraction = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

feat_extractor = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1)
feat_extractor.fc = torch.nn.Identity()
feat_extractor = feat_extractor.to(device).eval()

@torch.inference_mode()
def extract_features(dataloader):
    feats = []
    for x, _ in dataloader:
        x = x.to(device)
        f = feat_extractor(x)
        f = F.adaptive_avg_pool1d(f.unsqueeze(1), 1).squeeze(1)
        feats.append(f.cpu())
    return torch.cat(feats, dim=0)

# --------------------------
# Label reconstructed unlabeled set
# --------------------------
reconstructed_unlabeled_root = '/PATH/TO/UNLABELED_RECON'  # folder of recon images
batch_size = 32
alpha = 0.5  # weight between cosine and euclidean
tau = 0.80   # confidence threshold

ds_unlabeled = ReconstructedDataset(reconstructed_unlabeled_root, transform=transform_for_extraction)
dl_unlabeled = DataLoader(ds_unlabeled, batch_size=batch_size, shuffle=False, num_workers=2)

# Extract features
print("Extracting features for unlabeled reconstructed images...")
@torch.inference_mode()
def process_unlabeled():
    features_all = []
    paths_all = []
    for ims, paths in dl_unlabeled:
        feats = extract_features(DataLoader(list(zip(ims, paths)), batch_size=len(ims)))
        features_all.append(feats)
        paths_all.extend(paths)
    return torch.cat(features_all, dim=0), paths_all

features, img_paths = process_unlabeled()

# Compute similarities
S = combined_similarity(features.to(device), Tm, alpha=alpha)
max_sim, best_idx = torch.max(S, dim=1)

# Assign proxy labels (-1 if below threshold)
proxy_labels = torch.where(max_sim >= tau, best_idx, torch.tensor(-1).to(device))

# --------------------------
# Save filtered proxy-labeled dataset
# --------------------------
output_labeled_root = '/PATH/TO/PROXY_LABELED_RECON'
os.makedirs(output_labeled_root, exist_ok=True)

saved_count = 0
for path, lbl in zip(img_paths, proxy_labels.cpu().tolist()):
    if lbl < 0:
        continue
    dst_dir = os.path.join(output_labeled_root, str(lbl))
    os.makedirs(dst_dir, exist_ok=True)
    Image.open(path).convert('L').save(os.path.join(dst_dir, os.path.basename(path)))
    saved_count += 1

print(f"Saved {saved_count} proxy-labeled images to {output_labeled_root}")
